In [1]:
# imports
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy import signal
from scipy.signal import find_peaks

mne.set_log_level("WARNING")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.grid"] = True


In [2]:
# paths
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_raw_dir = project_root / "data" / "raw"
data_processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "outputs" / "figures"
qc_dir = project_root / "outputs" / "qc"

for folder in [data_raw_dir, data_processed_dir, figures_dir, qc_dir]:
    folder.mkdir(parents=True, exist_ok=True)

fif_path = data_raw_dir / "raw_artefacts_emg.fif"
annotations_path = qc_dir / "raw_base-annot.fif"
legacy_annotations_path = qc_dir / "raw_base_annotations.csv"

# Альтернативы для ручного запуска:
# fif_path = Path(r"/Users/user/Desktop/raw_artefacts_emg.fif")
# fif_path = Path(r"C:\\Users\\user\\Desktop\\raw_artefacts_emg.fif")

print("Project root:", project_root)
print("Input FIF:", fif_path)
print("Processed:", data_processed_dir)
print("Figures:", figures_dir)
print("QC:", qc_dir)


Project root: /Users/user/emg_artefacts_filter
Input FIF: /Users/user/emg_artefacts_filter/data/raw/raw_artefacts_emg.fif
Processed: /Users/user/emg_artefacts_filter/data/processed
Figures: /Users/user/emg_artefacts_filter/outputs/figures
QC: /Users/user/emg_artefacts_filter/outputs/qc


In [3]:
# clone raw
if not fif_path.exists():
    raise FileNotFoundError(f"Файл не найден: {fif_path}")

raw_original = mne.io.read_raw_fif(fif_path, preload=True)
raw_base = raw_original.copy()

# При повторном запуске восстанавливаем сохранённую разметку.
if annotations_path.exists():
    saved_annotations = mne.read_annotations(annotations_path)
    raw_base.set_annotations(saved_annotations)
    annotations_source = annotations_path
elif legacy_annotations_path.exists():
    # Однократная совместимость со старым CSV: MNE записал onset как дату от 1970-01-01.
    annotations_table = pd.read_csv(legacy_annotations_path)
    epoch = pd.Timestamp("1970-01-01", tz="UTC")
    annotation_onsets = (
        pd.to_datetime(annotations_table["onset"], utc=True) - epoch
    ).dt.total_seconds().to_numpy()
    saved_annotations = mne.Annotations(
        onset=annotation_onsets,
        duration=annotations_table["duration"].to_numpy(),
        description=annotations_table["description"].astype(str).to_numpy(),
        orig_time=None,
    )
    raw_base.set_annotations(saved_annotations)
    annotations_source = legacy_annotations_path
else:
    annotations_source = None

sfreq = raw_base.info["sfreq"]
duration_s = raw_base.n_times / sfreq

print("File:", fif_path.name)
print("Channels:", len(raw_base.ch_names))
print("sfreq:", sfreq)
print("Duration, s:", round(duration_s, 2))
print("First channels:", raw_base.ch_names[:10])
print("Annotations:", len(raw_base.annotations))
print("Annotations loaded from:", annotations_source or "input FIF")


/var/folders/sw/007c5hsj5zggxfp7nts4rq2w0000gn/T/ipykernel_9659/338245936.py:5: RuntimeWarning: This filename (/Users/user/emg_artefacts_filter/data/raw/raw_artefacts_emg.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_original = mne.io.read_raw_fif(fif_path, preload=True)


File: raw_artefacts_emg.fif
Channels: 8
sfreq: 4000.0
Duration, s: 117.52
First channels: ['RF R', 'BF R', 'TA R', 'GM R', 'RF L', 'BF L', 'TA L', 'GM L']


In [4]:
# check
channels_table = pd.DataFrame(
    {
        "channel": raw_base.ch_names,
        "type": raw_base.get_channel_types(),
        "bad": [ch in raw_base.info["bads"] for ch in raw_base.ch_names],
    }
)

display(channels_table)
print(f"Sampling rate: {raw_base.info['sfreq']} Hz")
print(f"Duration: {duration_s:.2f} s")
print("Bad channels:", raw_base.info["bads"] or "none")

raw_base.plot()


,channel,type,bad
0,RF R,eeg,False
1,BF R,eeg,False
2,TA R,eeg,False
3,GM R,eeg,False
4,RF L,eeg,False
5,BF L,eeg,False
6,TA L,eeg,False
7,GM L,eeg,False


Sampling rate: 4000.0 Hz
Duration: 117.53 s
Bad channels: none


In [ ]:
# Сохранение всех аннотаций в файл

raw_base.annotations.save(
    annotations_path,
    overwrite=True,
)

print(f"Сохранено аннотаций: {len(raw_base.annotations)}")
print("Файл:", annotations_path)

In [6]:
emg_channels = ["GM R", "GM L", "RF R", "RF L", "TA R", "TA L", "BF R", "BF L"]
right_channels = ["GM R", "RF R", "TA R", "BF R"]
left_channels = ["GM L", "RF L", "TA L", "BF L"]

excluded_channels = ["TA L"]

print("EMG channels:", emg_channels)
print("Right:", right_channels)
print("Left:", left_channels)
print("Excluded:", excluded_channels)


EMG channels: ['GM R', 'GM L', 'RF R', 'RF L', 'TA R', 'TA L', 'BF R', 'BF L']
Right: ['GM R', 'RF R', 'TA R', 'BF R']
Left: ['GM L', 'RF L', 'TA L', 'BF L']
Excluded: ['TA L']


In [7]:
# Global CAR

car_channels = [ch for ch in emg_channels if ch not in excluded_channels]

raw_car_global = raw_base.copy()

avg = raw_car_global.get_data(picks=car_channels).mean(axis=0)
idx = [raw_car_global.ch_names.index(ch) for ch in car_channels]

raw_car_global._data[idx] -= avg

car_global_path = data_processed_dir / "recording_car_global_raw.fif"
raw_car_global.save(car_global_path, overwrite=True)

print("Saved:", car_global_path)
raw_car_global.plot()


Saved: /Users/user/emg_artefacts_filter/data/processed/recording_car_global_raw.fif


In [8]:
# CAR отдельно справа и слева

raw_car_left_right = raw_base.copy()

for channels in [right_channels, left_channels]:
    car_channels = [
        ch for ch in channels
        if ch not in excluded_channels
    ]

    avg = raw_car_left_right.get_data(picks=car_channels).mean(axis=0)
    idx = [raw_car_left_right.ch_names.index(ch) for ch in car_channels]

    raw_car_left_right._data[idx] -= avg

car_left_right_path = (
    data_processed_dir / "recording_car_left_right_raw.fif"
)
raw_car_left_right.save(car_left_right_path, overwrite=True)

print("Saved:", car_left_right_path)

raw_car_left_right.plot()


Saved: /Users/user/emg_artefacts_filter/data/processed/recording_car_left_right_raw.fif


In [ ]:
# SVD: параметры и список каналов

SVD_LABEL = "artifact"  # Это имя ручных меток с примерами нужного артефакта.
SVD_APPLY_MODE = "mask"  # "mask" чистит только найденные артефакты, "whole" чистит всю запись.
SVD_SCORE_Z = 4.0  # Это порог похожести участка записи на первую SVD-компоненту.
SVD_PAD_MS = 5.0  # Это запас по времени вокруг найденного артефакта.
USE_SAVED_SVD_ANNOTATIONS = True  # Поставь True после перезапуска ноутбука, чтобы загрузить старую разметку.

svd_fit_channels = [  # Это хорошие EMG-каналы для поиска общего артефакта.
    ch for ch in emg_channels
    if ch not in excluded_channels
]

svd_annotations_path = qc_dir / "svd_artifact_training_annotations.csv"  # Здесь сохраняется ручная разметка артефактов.
svd_model_path = qc_dir / "svd_k1_model_and_mask.npz"  # Здесь сохраняются веса компоненты и маска очистки.
svd_output_path = data_processed_dir / "recording_svd_k1_masked_raw.fif"  # Это итоговая очищенная запись.

missing_channels = [  # Это проверка, что все указанные каналы есть в записи.
    ch for ch in svd_fit_channels
    if ch not in raw_base.ch_names
]

raw_svd_mark = raw_base.copy()  # Эта копия нужна только для ручной разметки артефактов.

if USE_SAVED_SVD_ANNOTATIONS and svd_annotations_path.exists():
    saved_annotations = mne.read_annotations(svd_annotations_path)  # Это загружает ранее сохранённые интервалы.
    raw_svd_mark.set_annotations(saved_annotations)  # Это переносит интервалы на рабочую копию.

print("SVD fit channels:", svd_fit_channels)
print("Excluded from SVD:", excluded_channels)
print("SVD mode:", SVD_APPLY_MODE)
print("Saved annotations found:", svd_annotations_path.exists())
